# 🌿 PlantGuard AI — 38-Class Model Training

**Version-locked to match local project exactly:**
| Package | Version |
|---|---|
| TensorFlow | 2.16.1 |
| Keras | 3.12.1 |
| NumPy | 1.26.4 |
| Pillow | 10.3.0 |
| h5py | 3.14.0 |

### Before you start
1. **Runtime → Change runtime type → T4 GPU** ✅
2. Get `kaggle.json` from [kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token
3. Run cells **one by one in order — do NOT skip the restart step**

## Step 1 — Pin exact versions & restart runtime
⚠️ **You MUST restart the runtime after this cell completes.**

In [ ]:
# Uninstall whatever Colab ships with first
!pip uninstall -y tensorflow tensorflow-gpu keras 2>/dev/null || true

# Install exact versions matching local requirements.txt
!pip install -q \
    'tensorflow==2.16.1' \
    'keras==3.12.1' \
    'numpy==1.26.4' \
    'pillow==10.3.0' \
    'h5py==3.14.0' \
    'kaggle'

print('\n✅ Packages installed.')
print('🔴 NOW: Runtime → Restart session, then continue from Step 2')

## Step 2 — Verify versions match local exactly
Run this after restarting. All versions must match the table above.

In [ ]:
import sys
import numpy as np
import PIL
import h5py
import tensorflow as tf
import keras

checks = {
    'Python'    : (sys.version.split()[0],  '3.'),
    'TensorFlow': (tf.__version__,           '2.16.1'),
    'Keras'     : (keras.__version__,        '3.12.1'),
    'NumPy'     : (np.__version__,           '1.26.4'),
    'Pillow'    : (PIL.__version__,          '10.3.0'),
    'h5py'      : (h5py.__version__,         '3.14.0'),
}

all_ok = True
for pkg, (got, want) in checks.items():
    ok = got.startswith(want) if pkg == 'Python' else got == want
    icon = '✅' if ok else '❌'
    print(f'{icon} {pkg:<12} got={got:<10} want={want}')
    if not ok: all_ok = False

assert all_ok, '❌ Version mismatch — re-run Step 1 and restart again'

gpus = tf.config.list_physical_devices('GPU')
assert gpus, '❌ No GPU — Runtime → Change runtime type → T4 GPU'
print(f'\n✅ GPU: {gpus[0].name}')
print('✅ All versions match — ready to train')

## Step 3 — Upload kaggle.json

In [ ]:
import os
from google.colab import files

print('👆 Click "Choose Files" and upload your kaggle.json')
files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ kaggle.json configured')

## Step 4 — Download & extract PlantVillage dataset (~1.5 GB)

In [ ]:
import os

print('📥 Downloading dataset...')
ret = os.system('kaggle datasets download -d vipoooool/new-plant-diseases-dataset')
assert ret == 0, '❌ Download failed — check your kaggle.json is valid'

print('📦 Extracting...')
os.system('unzip -q new-plant-diseases-dataset.zip -d plantdata')
print('✅ Done')

## Step 5 — Locate train / validation directories

In [ ]:
import os

TRAIN_DIR = VAL_DIR = None
for root, dirs, _ in os.walk('plantdata'):
    if 'train' in dirs and TRAIN_DIR is None:
        TRAIN_DIR = os.path.join(root, 'train')
    if 'valid' in dirs and VAL_DIR is None:
        VAL_DIR = os.path.join(root, 'valid')

assert TRAIN_DIR and os.path.isdir(TRAIN_DIR), '❌ train folder not found'
assert VAL_DIR   and os.path.isdir(VAL_DIR),   '❌ valid folder not found'

print(f'Train : {TRAIN_DIR}  ({len(os.listdir(TRAIN_DIR))} classes)')
print(f'Valid : {VAL_DIR}   ({len(os.listdir(VAL_DIR))} classes)')
print('\nAvailable classes:')
for c in sorted(os.listdir(TRAIN_DIR)):
    print(' ', c)

## Step 6 — Map all 38 classes
Exact order matches `CLASS_LABELS` in `backend/model.py` — **do not reorder**.

In [ ]:
import os, shutil, re

CLASS_LABELS_38 = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry___Powdery_mildew', 'Cherry___healthy',
    'Corn___Cercospora_leaf_spot', 'Corn___Common_rust', 'Corn___Northern_Leaf_Blight', 'Corn___healthy',
    'Grape___Black_rot', 'Grape___Esca', 'Grape___Leaf_blight', 'Grape___healthy',
    'Orange___Haunglongbing',
    'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy',
    'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites',
    'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus', 'Tomato___healthy',
]

def normalize(s):
    return re.sub(r'[^a-z0-9]', '', s.lower())

def find_match(cls, folder_list):
    if cls in folder_list: return cls
    nc = normalize(cls)
    for f in folder_list:
        if normalize(f) == nc: return f
    for f in folder_list:
        if nc in normalize(f) or normalize(f) in nc: return f
    return None

FILTERED_TRAIN = 'filtered/train'
FILTERED_VAL   = 'filtered/valid'
os.makedirs(FILTERED_TRAIN, exist_ok=True)
os.makedirs(FILTERED_VAL,   exist_ok=True)

avail_train = os.listdir(TRAIN_DIR)
avail_val   = os.listdir(VAL_DIR)
missing = []

for cls in CLASS_LABELS_38:
    mt = find_match(cls, avail_train)
    mv = find_match(cls, avail_val)
    if mt and mv:
        dst_t = os.path.join(FILTERED_TRAIN, cls)
        dst_v = os.path.join(FILTERED_VAL, cls)
        if not os.path.exists(dst_t): shutil.copytree(os.path.join(TRAIN_DIR, mt), dst_t)
        if not os.path.exists(dst_v): shutil.copytree(os.path.join(VAL_DIR,   mv), dst_v)
        print(f'✅ {cls:<45} train={len(os.listdir(dst_t)):>4}  val={len(os.listdir(dst_v)):>3}')
    else:
        missing.append(cls)
        print(f'❌ NOT FOUND: {cls}')

print(f'\n{38 - len(missing)}/38 classes ready')
if missing:
    raise RuntimeError(f'Missing classes: {missing}')

## Step 7 — Build data pipelines

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH    = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_TRAIN,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=True,
    class_names=CLASS_LABELS_38,
    seed=42,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    FILTERED_VAL,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    label_mode='categorical',
    shuffle=False,
    class_names=CLASS_LABELS_38,
)

# MobileNetV2 expects pixels in [-1, 1] — use the official preprocess function
def preprocess(x, y):
    return tf.keras.applications.mobilenet_v2.preprocess_input(x), y

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess,   num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

print(f'✅ Train batches : {len(train_ds)}')
print(f'✅ Val batches   : {len(val_ds)}')

## Step 8 — Build MobileNetV2 model (38 classes)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

NUM_CLASSES = 38
IMG_SIZE    = 224

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base.trainable = False

# Augmentation on GPU — same transforms used in ensemble inference
augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),
], name='augmentation')

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = augment(inputs, training=True)
x       = base(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'✅ Output shape      : {model.output_shape}')
print(f'✅ Trainable params  : {model.count_params():,}')
print(f'✅ TF version in use : {tf.__version__}')

## Step 9 — Phase 1: Train top layers, base frozen (15 epochs)

In [ ]:
callbacks = [
    # Save using the legacy .h5 format — matches what backend/model.py loads
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_format='h5',
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True,
        monitor='val_accuracy',
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.3, patience=2,
        monitor='val_loss', verbose=1,
    ),
]

print('🚀 Phase 1: Training top layers (base frozen)...')
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=callbacks,
)
print(f'\n✅ Phase 1 done — best val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## Step 10 — Phase 2: Fine-tune top 50 base layers (15 more epochs)

In [ ]:
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

print(f'Fine-tuning {sum(1 for l in base.layers if l.trainable)} base layers')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print('🚀 Phase 2: Fine-tuning...')
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=callbacks,
)
print(f'\n✅ Phase 2 done — best val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## Step 11 — Evaluate & per-class accuracy report

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

loss, acc = model.evaluate(val_ds, verbose=1)
print(f'\n🎯 Final Validation Accuracy : {acc*100:.2f}%')
print(f'📉 Final Validation Loss     : {loss:.4f}')

# Training curve
acc_all = history1.history['accuracy']     + history2.history['accuracy']
val_all = history1.history['val_accuracy'] + history2.history['val_accuracy']
split   = len(history1.history['accuracy'])

plt.figure(figsize=(11, 4))
plt.plot(acc_all, label='Train', color='#2d6a4f')
plt.plot(val_all, label='Val',   color='#52b788')
plt.axvline(x=split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Training Accuracy — 38 Classes')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.tight_layout(); plt.show()

# Per-class accuracy
print('\n📊 Per-class accuracy (⚠️ = below 80%):')
y_true, y_pred = [], []
for x_batch, y_batch in val_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)
for i, cls in enumerate(CLASS_LABELS_38):
    mask = y_true == i
    if mask.sum() > 0:
        cls_acc = (y_pred[mask] == i).mean()
        bar  = '█' * int(cls_acc * 20)
        flag = ' ⚠️' if cls_acc < 0.80 else ''
        print(f'  {cls:<45} {cls_acc*100:5.1f}%  {bar}{flag}')

## Step 12 — Save & verify the model
Loads the best checkpoint and verifies output shape and format are correct for your backend.

In [ ]:
import numpy as np
import tensorflow as tf

# Load the best checkpoint (saved in h5 format during training)
best = tf.keras.models.load_model('best_model.h5')

# Verify output shape
dummy = np.zeros((1, 224, 224, 3), dtype='float32')
out   = best.predict(dummy, verbose=0)

assert out.shape == (1, 38),          f'❌ Wrong output shape: {out.shape} — expected (1, 38)'
assert abs(out[0].sum() - 1.0) < 1e-4, f'❌ Probs do not sum to 1: {out[0].sum()}'

# Verify it loads with the same preprocessing your backend uses
test_input = tf.keras.applications.mobilenet_v2.preprocess_input(
    np.random.randint(0, 255, (1, 224, 224, 3)).astype('float32')
)
out2 = best.predict(test_input, verbose=0)
assert out2.shape == (1, 38), '❌ Preprocessing test failed'

print(f'✅ Output shape  : {out.shape}')
print(f'✅ Prob sum      : {out[0].sum():.6f}')
print(f'✅ TF version    : {tf.__version__}')
print(f'✅ Top class idx : {int(np.argmax(out2[0]))} → {CLASS_LABELS_38[int(np.argmax(out2[0]))]}')
print('\n✅ Model verified — ready to download')

## Step 13 — Download

After downloading, place the file here in your local project:
```
backend/model/best_model.h5
```
Then restart uvicorn — no other changes needed.

In [ ]:
from google.colab import files
files.download('best_model.h5')
print('✅ Place in: backend/model/best_model.h5')
print('✅ Then restart uvicorn')